## Required packages

The version of the required packages is in "requirements.txt" file.

In [1]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from pylab import *

import keras.optimizers
import tensorflow
tensorflow.random.set_seed(42)

from tensorflow import keras
from tensorflow.keras.layers import Input, Masking, Dense, Dropout, LSTM, RepeatVector, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model, Sequential
from keras_preprocessing.sequence import pad_sequences
from tensorflow.keras import initializers
from tensorflow.keras import backend as K

import os

2025-08-20 16:04:02.390455: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Functions to read, train and test data

This section contains the necessary functions, both to read the csv files and to tranform the size of the data so that they are suitable for the LSTM Autoencoder to train and obtain the predictions.

In [2]:
def reading_files(path):
    
    """
    Read csv files, drop 'Unnamed: 0' column and set 'UtcTime' as index.
    
    Args:
        path (str): Preprocessed train-test input files.
        
    Returns:
        df (pandas.DataFrame): Preprocessed DataFrame.
    """
    
    df = pd.read_csv(path)
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    
    if 'Attack' in df.columns:
        df.Attack = df.Attack.astype(str)
    df=df.reset_index(drop=True)
    
    return df

In [3]:
def timesteps_calculation(df, timesteps_max):

    timesteps_calc = int(pd.Series(df.ProcessGuid).value_counts().quantile(0.5))
    if timesteps_calc >= timesteps_max:
        timesteps = timesteps_max
    else:
        timesteps = timesteps_calc

    print("timesteps:", timesteps)

    return timesteps

In [4]:
def data_generator(input_data, batch_size, timesteps, n_batches):
    
    """
    Generates batches of training data for LSTM Autoencoder model.
    
    Args:
        input_data (ndarray): Input data array of shape (samples, sequence_length, features).
        batch_size (int): Size of each batch.
        timesteps (int): Desired length of each sequence in a batch.
        
    Yields:
        tuple: A tuple containing the input batch and the corresponding target batch.
    
    """
    input_data = input_data[:n_batches * batch_size]
    while True:
        for batch in range(0, n_batches):
            current_batch = input_data[batch * batch_size:(batch + 1) * batch_size]
            current_batch = pad_sequences(current_batch, maxlen=timesteps)

            yield current_batch, current_batch

In [5]:
def reshaping_data(X, timesteps, test):
    
    """
    Reshapes the input data into a suitable format for LSTM models.
    
    Args:
        X (ndarray): Input data array of shape (n_samples, features).
        timesteps (int): Number of time steps or sequence length for the reshaped data.
        test (bool): Specify whether to reshape the tet or training dataset. If test=True it will be reshaped, otherwise train.
        
    Returns:

        ndarray: Reshaped input data array of shape (n_samples - timesteps + 1, timesteps, features).
    
    """
    
    if test:
        Xs = pd.DataFrame()
        for i in X['ProcessGuid'].unique():
            sorteddf=X[X.ProcessGuid == i].sort_values(by="UtcTime")
            temp_col = sorteddf["UtcTime"]
            Attack = sorteddf["Attack"]
            matrix_temporal = sorteddf.drop(["ProcessGuid","UtcTime","Attack","collector_node_id"], axis=1).values # "collector-node-id"

            # Define padding values and amount
            padding_value = -1  # Change this to the value you want for padding
            padding_rows = timesteps - 1  # Number of rows to add as padding

            # Pad the matrix along the rows
            padded_matrix = np.pad(matrix_temporal, ((padding_rows, 0), (0, 0)), mode='constant', constant_values=padding_value)

            for j,z1,z2 in zip(range(len(matrix_temporal) - timesteps + 1),temp_col,Attack): # Ensures that extracted substrings have uniform length of timesteps and do not go outside the original sequence boundary. Avoid extracting incomplete substring
                data = {'Value': [padded_matrix[j:(j + timesteps)]],
                        'ProcessGuid': [i],
                        'UtcTime':z1,
                        "numVentana":j,
                        "Attack":z2}
                df = pd.DataFrame(data)
                Xs=pd.concat([Xs,df])
        return Xs
    else:
        Xs = []
        for i in X.ProcessGuid.unique():
            matrix_temporal = X[X.ProcessGuid == i].sort_values(by="UtcTime").drop(["ProcessGuid","UtcTime"], axis=1).values

            # Define padding values and amount
            padding_value = -1  # Change this to the value you want for padding
            padding_rows = timesteps - 1  # Number of rows to add as padding
            
            # Pad the matrix along the rows
            padded_matrix = np.pad(matrix_temporal, ((padding_rows, 0), (0, 0)), mode='constant', constant_values=padding_value)
            for j in range(len(matrix_temporal) - timesteps + 1): # Ensures that extracted substrings have uniform length of timesteps and do not go outside the original sequence boundary. Avoid extracting incomplete substring
                Xs.append(padded_matrix[j:(j + timesteps)])
                

        return np.array(Xs)

In [6]:
def add_layer(layer_conf, x_input, X_train_shape=None):

    """
    Adds a layer to the network based on the provided configuration.

    Args:
        layer_conf (dict): Dictionary containing the configuration of the layer.
            The keys in the dictionary can include:
                - "type" (str): Type of the layer (e.g., 'lstm', 'dense', 'dropout', etc.).
                - "size" (int): Number of units or neurons in the layer (for 'lstm', 'dense', etc.).
                - "activation" (str): Activation function to use (e.g., 'relu', 'sigmoid').
                - "return-sequences" (bool, optional): Whether to return sequences in LSTM layers.
                - "init" (str, optional): Kernel initializer scheme (default is 'glorot_uniform').
                - "recurrent-dropout" (float, optional): Dropout rate for recurrent units (for 'lstm', 'bidirectional').
                - "rate" (float, optional): Dropout rate (for 'dropout' layer).
        
        x_input (Tensor): Input tensor for the layer.
        
        X_train_shape (tuple, optional): Shape of the training data, used in certain layers like
            'repeat-vector' and 'time-distributed'. If required by the layer, it should be provided 
            as (n_samples, timesteps, features).
        
    Returns:
        Tensor: The output tensor after applying the specified layer.
    """
    
    init_scheme = layer_conf.get("init", "glorot_uniform") # Default to 'glorot_uniform'

    if layer_conf["type"] == "lstm":
        return LSTM(units=layer_conf["size"], activation=layer_conf["activation"],
                    return_sequences=layer_conf["return-sequences"],
                    kernel_initializer=initializers.get(init_scheme),
                    recurrent_dropout=layer_conf.get("recurrent-dropout", 0))(x_input) 

    elif layer_conf["type"] == "dense":
        return Dense(units=layer_conf["size"], activation=layer_conf["activation"],
                     kernel_initializer=initializers.get(layer_conf["init"]))(x_input)

    elif layer_conf["type"] == "dropout":
        return Dropout(rate=layer_conf["rate"])(x_input)
    elif layer_conf["type"] == "repeat-vector":
        return RepeatVector(X_train_shape[1])(x_input)
    elif layer_conf["type"] == "bidirectional":
        return Bidirectional(LSTM(units=layer_conf["size"], activation=layer_conf["activation"],
                                  return_sequences=layer_conf["return-sequences"],
                                  recurrent_dropout=layer_conf["recurrent-dropout"],
                                  kernel_initializer=initializers.get(init_scheme)))(x_input)

    elif layer_conf["type"] == "time-distributed":
        return TimeDistributed(Dense(X_train_shape[2], activation="sigmoid",
                                     kernel_initializer=initializers.get(init_scheme)))(x_input)

In [7]:
def custom_binary_crossentropy(y_true, y_pred):
    
    """
    Custom binary crossentropy loss function that applies a mask to ignore certain target values.
    
    Args:
        y_true (Tensor): Ground truth binary labels.
        y_pred (Tensor): Predicted probabilities.
    
    Returns:
        Tensor: The mean binary crossentropy loss, excluding masked values (those where y_true is -1).
    """
    
    mask = K.cast(K.not_equal(y_true, -1), K.floatx())
    y_true = K.cast(y_true, K.floatx())  
    loss = K.binary_crossentropy(y_true, y_pred) * mask
    return K.sum(loss) / K.sum(mask)

In [8]:
def lstm_autoencoder_training(timesteps, X_train, model_conf):
    
    """
    Trains the LSTM Autoencoder model according to the configured parameters.
    
    Args:
        timesteps (int): Length of each sequence in a batch.
        X_train (ndarray): Input training data array (normal data, no attacks) of shape (samples, sequence_length, features).
        model_conf (dict): Configurable model parameters.
        
    Returns:
        lstm_autoencoder (keras.engine.training.Model): LSTM Autoencoder trained model.
        threshold (float): Minimum anomaly level.
    """

    input_shape = (X_train.shape[1], X_train.shape[2])
    initial_seq = Sequential()
    initial_seq.add(Input(shape=input_shape))
    initial_seq.add(Masking(mask_value=-1)) # Must match padding_value 
    input_seq = initial_seq.input
    x = initial_seq.output

    lstm_autoencoder_conf = model_conf["lstm-autoencoder"]

    encoder_conf = lstm_autoencoder_conf["encoder"]

    # Encoder
    for i in range(encoder_conf["n-layers"]):
        layer = layer = encoder_conf[str(i)]
        x = add_layer(layer, x, X_train.shape if layer["type"] in ["repeat-vector", "time-distributed"] else None)

    # Decoder
    decoder_conf = lstm_autoencoder_conf["decoder"]

    for j in range(decoder_conf["n-layers"]):
        layer = decoder_conf[str(j)]
        x = add_layer(layer, x, X_train.shape if layer["type"] in ["time-distributed"] else None)

    output = TimeDistributed(Dense(input_shape[-1], activation="sigmoid"))(x)
    lstm_autoencoder = Model(inputs=input_seq, outputs=output)

    if lstm_autoencoder_conf["optimizer"] == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=lstm_autoencoder_conf["learning-rate"])

    lstm_autoencoder.compile(optimizer=optimizer, loss=custom_binary_crossentropy)
    
    # Number of batches depending on the input size and batch_size, rounding down to make sure that no samples are left out
    n_batches = len(X_train) // lstm_autoencoder_conf["batch-size"]

    early_stopping = EarlyStopping(monitor='loss', patience=10, verbose=1, restore_best_weights=True,
                                   min_delta=0.001, mode='min')
 
    data_gen = data_generator(batch_size=lstm_autoencoder_conf["batch-size"], timesteps=timesteps, input_data=X_train, n_batches=n_batches)

    lstm_autoencoder.fit(x=data_gen, steps_per_epoch=n_batches, epochs=lstm_autoencoder_conf["epochs"],callbacks=[early_stopping])
    
    X_train = X_train.astype('float32')

    pred_train = lstm_autoencoder.predict(X_train)
    absolute_errors = np.abs(X_train - pred_train)
    mask = (X_train != -1.0).astype(np.float32)
    absolute_errors[X_train == -1.0] = 0.0
    mae = np.sum(absolute_errors, axis=(1, 2)) / np.sum(mask, axis=(1, 2))
    X_train_df=pd.DataFrame()
    X_train_df["error"] = mae

    th2=X_train_df.error.apply("mean")+X_train_df.error.apply("std")*2.5
    th1=np.percentile(X_train_df.error, 99)
    threshold = max(th1, th2)

    return lstm_autoencoder, th1

In [9]:
def lstm_autoencoder_test_prediction(lstm_autoencoder, X_test):
    
    """
    Predicts the input values of the training and test data, storing the error obtained for each of them.
    
    Args:
        lstm_autoencoder (keras.engine.training.Model): Length of each sequence in a batch.
        X_test (DataFrame): Input test data frame.   
    Returns:
        X_test (DataFrame): Input test data frame.
    """

    X_test_values = np.array(X_test["Value"].tolist())
    X_test_values = X_test_values.astype("float32")
    predictions = lstm_autoencoder.predict(X_test_values)
    
    return predictions, X_test_values

In [10]:
def processing_errors(X_test_values, predictions, X_test):
    
    """
    Processes the errors between true values and predictions, calculating the Mean Absolute Error (MAE).

    Args:
        X_test_values (ndarray): Array containing the true values (ground truth). 
                Values equal to -1.0 are treated as missing and will be ignored.
        predictions (ndarray): Array containing the predicted values from the model.
        X_test (DataFrame): Pandas DataFrame that stores additional features and will be updated with the error metrics.
    
    Returns:
        X_test (DataFrame): Updated DataFrame with two new columns:
            - 'absolute_error': The absolute error for each element.
            - 'mae': The Mean Absolute Error (MAE) for each sample, ignoring values where true values are -1.0.
        absolute_error (ndarray): The absolute error array after processing.
    """
    
    mask = (X_test_values != -1.0).astype(np.float32)

    absolute_error = np.abs(X_test_values - predictions)  # absolute error
    absolute_error[X_test_values == -1.0] = 0.0
    X_test['absolute_error'] = list(absolute_error)
    X_test['mae'] = np.sum(absolute_error, axis=(1, 2)) / np.sum(mask, axis=(1, 2))  # MAE
    return X_test, absolute_error

### Plots Functions

In [11]:
def plot_sequence_errors(datashaped, mean_error):
    
    """
    Plots a scatter plot of the error for each sequence, with points connected by 'ProcessGuid' and symbols based on 'Attack' labels.

    Args:
        datashaped (DataFrame): A pandas DataFrame containing the data to plot, including columns for UtcTime, numVentana, ProcessGuid, Attack, and the error metrics.
        mean_error (str): The error metric to plot. It determines the y-axis label and plot configuration.
    
    Returns:
        None: The function directly displays the plot.
    """
    
    labels = {"mae": "MAE", "UtcTime": "UtcTime"}

    markers = {
        '0': 'circle',  
        '1': 'x'  
    }
    fig = px.scatter(datashaped, x='UtcTime', y=mean_error,
                        hover_data=["numVentana"],
                        color='ProcessGuid',  
                        symbol='Attack',  
                        symbol_map=markers,  
                        labels=labels,
                        title=f"{mean_error} by Sequence connected by ProcessGuid")

    fig.add_hline(y=datashaped['threshold'].iloc[0], line_dash="dash", line_color="red",
                    annotation_text="Anomaly Threshold")
    fig.update_traces(marker=dict(size=7), selector=dict(mode='markers')) 
    fig.update_layout(legend_title_text='ProcessGuid, Attack')  
    fig.show()

In [12]:
def remove_params_by_error_filter(percentile, data_for_plot):
    
    """
    Removes columns from a DataFrame where none of the values exceed the given percentile threshold.

    Args:
        percentile (float): Percentile value (0-100) used to determine the threshold. Columns are removed if no value in the column exceeds this threshold.
        data_for_plot (DataFrame): Pandas DataFrame containing the data to filter. Each column represents a different parameter or feature.
    
    Returns:
        DataFrame: A new DataFrame where columns that do not exceed the given percentile threshold have been removed.
    """
    
    data_for_plot_result = data_for_plot
    percentile = np.percentile(data_for_plot, percentile)
    for column in data_for_plot.columns:
        filter_data = pd.unique((data_for_plot[column] > percentile).values.ravel())
        if True not in filter_data:
            data_for_plot_result = data_for_plot_result.drop(column, axis=1)
    return data_for_plot_result

def plots_heatmap_cols_anom_seq(df, selected_col, inf_cols_test, mean_error, error, percentile):
    
    """
    Plots a heatmap of the average reconstruction error per feature for anomalous sequences in a dataset.

    Args:
        df (DataFrame): A pandas DataFrame containing the data, including columns for anomalies, errors, and the selected feature.
        selected_col (str): The name of the column used to group the anomalies (e.g., 'ProcessGuid', 'SequenceID').
        inf_cols_test (list): A list of feature column names to be used for calculating errors.
        mean_error (str): The error metric to display ('mae').
        error (str): The column in the DataFrame that contains the error values.
        percentile (float): The percentile threshold used to filter out features with insignificant errors.
    
    Returns:
        None: The function displays a heatmap for each unique value in the `selected_col` where anomalies are present.
    """
    
    for value in df[selected_col].unique():
        anomalies = df[(df["anomaly"] == True) & (df[selected_col] == value)]
        if len(anomalies) > 0:
            errors_to_plot = np.array([row for row in anomalies[error]])
            mean_errors_per_feature = np.mean(errors_to_plot, axis=1)

            data_for_plot = pd.DataFrame(mean_errors_per_feature, columns=inf_cols_test, index=anomalies["UtcTime"])
            data_for_plot = remove_params_by_error_filter(percentile, data_for_plot=data_for_plot)

            dataframe = data_for_plot.T
            dataframe.index = [column[:50] if len(column) > 50 else column for column in data_for_plot.columns]
            if data_for_plot.index.nunique() > 1:
                fig = px.imshow(dataframe,
                                labels=dict(x="UtcTime of Anomalous Sequences", y="Features", color=mean_error),
                                x=anomalies["UtcTime"],
                                y=dataframe.index,
                                aspect="auto",
                                title=f"HeatMap",
                                color_continuous_scale="YlOrBr")
                # Actualizar el layout para mejorar la visualización
                fig.update_layout(
                    title={
                        "text": f"HeatMap of the Average Reconstruction Error per Feature in Anomalous Sequences - {value}",
                        'y': 0.95,
                        'x': 0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'
                    },
                    yaxis={'tickmode': 'array', 'tickvals': np.arange(len(inf_cols_test)),
                            'ticktext': inf_cols_test},
                    autosize=True,
                    margin=dict(t=100, l=10, r=10, b=10))
                fig.update_traces(
                    xgap=1,  
                    ygap=1,  
                    showscale=True  
                )
                fig.update_layout(
                    autosize=True,
                    width=None,  
                    height=None,  
                )
                
                fig.show()

In [13]:
def plot_median_errors_by_processGuid(datashaped, error_name):
    
    """
    Plots the median error for each sequence identified by 'ProcessGuid' over time, with symbols indicating the presence of attacks.

    Args:
        datashaped (DataFrame): A pandas DataFrame containing the processed data. It should include the following columns:
            - 'UtcTime': Timestamp for each sequence.
            - 'median_error_by_seq': Median error for each sequence ('mae').
            - 'ProcessGuid': Identifier for each process sequence.
            - 'Attack': Indicator of whether an attack occurred (0 or 1).
            - 'threshold': Anomaly threshold for comparison.
        error_name (str): The name of the error metric being plotted. This is used for labeling the y-axis and the title.
    
    Returns:
        None: The function generates and displays an interactive scatter plot.
    """
    
    markers = {
        '0': 'circle',  
        '1': 'x'  
    }
    fig = px.scatter(datashaped, x='UtcTime', y='median_error_by_seq',
                        color='ProcessGuid', 
                        symbol='Attack', 
                        symbol_map=markers,  
                        labels={
                            "median_error_by_seq": f"Median error - {error_name} - By ProcessGuid",
                            "UtcTime": "UtcTime"
                        },
                        title=f"{error_name} by ProcessGuid")

    fig.add_hline(y=datashaped['threshold'].iloc[0], line_dash="dash", line_color="red",
                    annotation_text="Anomaly Threshold")
    fig.update_traces(marker=dict(size=7), selector=dict(mode='markers'))  
    fig.update_layout(legend_title_text='ProcessGuid, Attack') 

    fig.show()

In [14]:
def all_plots_by_error_name(X_test_errors, percentile, threshold, X_test):
    
    """
    Generates various plots for analyzing errors and anomalies in the test dataset based on specified thresholds and error metrics.

    Args:
        X_test_errors (DataFrame): DataFrame containing the error values for the test dataset, including columns for 'mae' and 'absolute_error'.
        percentile (float): Percentile value used for filtering features based on their error values.
        threshold (float): Threshold value used to determine whether a sequence is considered anomalous.
        X_test (DataFrame): Original test dataset that contains both informative columns (e.g., 'ProcessGuid', 'UtcTime') and features to be evaluated.

    Returns:
        None: The function generates and displays various plots:
            - A sequence error plot for anomalous sequences.
            - A heatmap of the average reconstruction error per feature in anomalous sequences.
            - A plot showing the median errors by 'ProcessGuid'.
    """
    
    inf_var_test = ["ProcessGuid", "UtcTime", "Attack", "collector_node_id"]
    cols_list = X_test.drop(inf_var_test, axis=1).columns.tolist()
    df_anomalies = pd.DataFrame()
    df_anomalies = pd.concat([df_anomalies, X_test_errors])
    error = 'absolute_error'
    mean_error = 'mae'
    df_anomalies['threshold'] = threshold
    df_anomalies['anomaly'] = df_anomalies[mean_error] > df_anomalies['threshold']

    if len(df_anomalies[df_anomalies.anomaly == True]) > 0:
        plot_sequence_errors(df_anomalies, mean_error)
        plots_heatmap_cols_anom_seq(df_anomalies, "ProcessGuid", cols_list, mean_error, error, percentile)

        df_anomalies = df_anomalies.reset_index(drop=True)

        df_anomalies_by_pg = pd.DataFrame()
        df_anomalies_by_pg["median_error_by_seq"] = df_anomalies.groupby('ProcessGuid')[mean_error].median()
        df_anomalies_by_pg["anomaly"] = df_anomalies.groupby('ProcessGuid')['anomaly'].any()
        unique_values = df_anomalies.groupby('ProcessGuid').agg({
            'UtcTime': 'first',
            'Attack': 'first',
            'threshold': 'first'
        }).reset_index()
        df_anomalies_by_pg = df_anomalies_by_pg.merge(unique_values, on='ProcessGuid', how='left')
        plot_median_errors_by_processGuid(df_anomalies_by_pg, mean_error)

### Configurable parameters

This is the section where all the necessary parameters are configured: input file paths, model parameters and anomaly report parameters, so that the number of anomalies obtained can be adjusted as desired.

In [20]:
# Input data foler
input_training_data_folder_path_c1=r"../Data/id17/s1_sim/training/csc.exe/data/" 
input_training_data_folder_path_c2=r"../Data/id17/s2_sim/training/csc.exe/data/" 
input_training_data_folder_path_c3=r"../Data/id17/v1_sim/training/csc.exe/data/" 
input_training_data_folder_path_c4=r"../Data/id17/v2_sim/training/csc.exe/data/" 
input_test_data_folder_path = r"../Data/id17/s2_sim_test/APT-C-35/csc.exe/data/" 

# Maximum sequence length to be used by the model in training for each batch. The final timesteps parameter will be computed
# later depending on the input data length
timesteps_max = 10

#Name of the clean label
label_clean_data = 0

# LSTM Autoencoder model configuration
model_conf = {
    "lstm-autoencoder": {
        "encoder": 
            {
                "n-layers": 5,
                "0": {"type": "lstm", "size":32, "activation": "tanh", "return-sequences":True},
                "1": {"type": "dropout", "rate":0.2},
                "2": {"type": "lstm", "size":8, "activation": "tanh", "return-sequences":False},
                "3": {"type": "dropout", "rate":0.2},
                "4": {"type": "repeat-vector"}
            },
        "decoder": 
            {
                "n-layers": 5,
                "0": {"type": "lstm", "size":8, "activation": "tanh", "return-sequences":True},
                "1": {"type": "dropout", "rate":0.2},
                "2": {"type": "lstm", "size":32, "activation": "tanh", "return-sequences":True},
                "3": {"type": "dropout", "rate":0.2},
                "4": {"type": "time-distributed"}
            },
        "optimizer": "adam",
        "learning-rate": 0.001,
        "loss": "mse",
        "epochs": 50,
        "batch-size": 25
    }
}


# Percentile
heatmap_percentile = 80

### Run

Finally, once the necessary packages have been downloaded and imported and the functions and configurable parameters have been loaded, it is time to execute the following cells in order. Here the different functions defined above are called with the necessary parameters, depending on what you want to obtain. 

In [23]:
# Reading input files 

X_train_c1 = reading_files(os.path.join(input_training_data_folder_path_c1,"training.csv"))
X_train_c2 = reading_files(os.path.join(input_training_data_folder_path_c2,"training.csv"))
X_train_c3 = reading_files(os.path.join(input_training_data_folder_path_c3,"training.csv"))
X_train_c4 = reading_files(os.path.join(input_training_data_folder_path_c4,"training.csv"))
X_test = reading_files(os.path.join(input_test_data_folder_path,"X_test.csv"))
X_test = X_test.drop(["index"], axis=1)


In [24]:
print(X_train_c1.shape)
print(X_train_c2.shape)
print(X_train_c3.shape)
print(X_train_c4.shape)
print(X_test.shape)

(122, 33)
(576, 33)
(229, 33)
(524, 33)
(65, 186)


In [ ]:
training = reading_files(os.path.join(input_training_data_folder_path,"training.csv"))
test = reading_files(os.path.join(input_test_data_folder_path,"test.csv"))

print(training.shape, test.shape)

In [ ]:
# Final timesteps depending of the X_train length. This is necessary because the length of the input data sequence is variable, 
# each process has a certain number of rows

timesteps = timesteps_calculation(X_train, timesteps_max)

In [ ]:
# Reshaping data so that the model can be trained

X_train = reshaping_data(X_train, timesteps=timesteps,test=False)
X_test_reshaping = reshaping_data(X_test, timesteps=timesteps,test=True)

print("\nTraining data shape:", X_train.shape)
print("Validation data shape:", (np.array(X_test_reshaping["Value"].tolist())).shape, "\n")

In [ ]:

X_train_y = np.zeros(X_train.shape[0])


from sklearn.model_selection import train_test_split
data_train, data_test, labels_train, labels_test = train_test_split(X_train, X_train_y, test_size=0.10, shuffle=False)

In [ ]:
data_train.shape

## Generate Synthetic data

In [ ]:
from util.TrainRoutine import AutoEncTrainRoutine
from util.aedpmerf import AEDPMERF

emb_dim = 12
eps = 1
aedpmerf = AEDPMERF(seq_len=timesteps, n_feat=X_train.shape[2], emb_dim=emb_dim, is_priv=True)

In [ ]:
model, history = aedpmerf.ae.train_model(data_train, labels_train, data_test, labels_test)
plt.plot(history["train"])
plt.plot(history["val"])
   
aedpmerf.save_ae("normal_train.pth")

In [ ]:
eps = 1
lr = 1e-3
n_epochs = 2000
enc_df = aedpmerf.encode_train_data(data_train, labels_train, fname=f"data/normal_training_encoded_embed{emb_dim}_eps{eps}.csv")

In [ ]:
enc_df.shape

In [ ]:
aedpmerf.train_gen(data=enc_df, mini_batch_size=0.1, lr=lr, eps=eps, n_epochs=n_epochs)
n_gen_samples = enc_df.shape[0]

In [ ]:
gen_data = aedpmerf.generate(n_gen_samples, fname=f"enc_gen_priv_embed{emb_dim}_eps{eps}.csv")

In [ ]:
gen_data[0].shape

In [ ]:
gen_data = np.array(gen_data)
gen_data = np.transpose(gen_data, (1, 0, 2))

In [ ]:
gen_data.shape

In [ ]:
# LSTM Autoencoder training

lstm_autoencoder_model, threshold = lstm_autoencoder_training(timesteps=timesteps,
                                                              X_train=gen_data,
                                                              model_conf=model_conf)

In [ ]:
lstm_autoencoder_model.summary()

In [ ]:
threshold

In [ ]:
# Error prediction and associated errorss
predictions, X_test_values= lstm_autoencoder_test_prediction(lstm_autoencoder_model, X_test_reshaping)
X_test_errors, absolute_error = processing_errors(X_test_values, predictions, X_test_reshaping)

In [ ]:
X_test_errors.shape

In [ ]:
X_test_errors.sort_values("mae", ascending=False)

### Plot results

In [ ]:
all_plots_by_error_name(X_test_errors, heatmap_percentile, threshold, X_test)

In [ ]:
def test_plot(X_test,threshold,label_clean_data):
    """
    Generates and plots both a confusion matrix and a scatter plot that visualizes the time-series anomaly scores.
    
    Args:
        X_test (DataFrame): Input test data frame.
        threshold (float): Minimum anomaly level.
    Returns:
        None
    """
    ## Scatter plot
    # Create the scatter plot
    fig = px.scatter(X_test, y="mae", x="UtcTime", color="Attack", hover_data=["ProcessGuid","UtcTime"])

    # Create data for the new line trace
    """
    x_line = X_test.index
    y_line = [threshold] * len(x_line)  # Replace [0] with your desired y-values for the line

    # Add the line trace to the existing figure
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', name='LinePerc'))
    
    """
    horizontal_line_value = threshold  # Adjust this value as needed

    # Add a horizontal line to the plot
    fig.add_shape(
        go.layout.Shape(
            type="line",
            x0=min(X_test['UtcTime']),
            x1=max(X_test['UtcTime']),
            y0=horizontal_line_value,
            y1=horizontal_line_value,
            line=dict(color="red", width=2)  # Customize line properties
        )
    )
    fig.show()

    ## CM plot
    dfcm=X_test.groupby("ProcessGuid").agg({"mae": "max", "Attack": "first"})
    TP=len(dfcm[(dfcm.Attack!=label_clean_data)& (dfcm.mae>threshold)])
    TN=len(dfcm[(dfcm.Attack==label_clean_data)& (dfcm.mae<=threshold)])
    FP=len(dfcm[(dfcm.Attack==label_clean_data)& (dfcm.mae>threshold)])
    FN=len(dfcm[(dfcm.Attack!=label_clean_data)& (dfcm.mae<=threshold)])

    print(TP)
    print(TN)
    print(FP)
    print(FN)

    if TP+FN == 0:
        TPrate=0
        FNrate=0
    else:
    
        TPrate=TP/(TP+FN)
        FNrate=FN/(TP+FN)

    if TN+FP == 0:
        TNrate=0
        FPrate=0
    else:
        TNrate=TN/(TN+FP)
        FPrate=FP/(TN+FP)

    print("Accuracy", (TP+TN)/(TP+TN+FP+FN))
    cm=[[FNrate,TNrate],[TPrate,FPrate]]
    infmetrix=[["False Negative rate","True Negative rate"],["True Positive rate","False Positive rate"]]

    # Create a heatmap
    trace = go.Heatmap(z=cm, 
                    y=["Predicted 0", "Predicted 1"],
                    x=["Actual 1", "Actual 0"],
                    text=infmetrix,
                    texttemplate="%{text}",
                    colorscale="sunsetdark")

    layout = go.Layout(title="Confusion Matrix per ProcessGUID",
                    xaxis=dict(title="Actual"),
                    yaxis=dict(title="Predicted"),
                    height=600,  # Set the height of the plot
                    width=800)    # Set the width of the plot)

    fig = go.Figure(data=[trace], layout=layout)

    # Display the confusion matrix
    fig.show()
    return None

In [ ]:
test_plot(X_test_errors,threshold,str(label_clean_data))